# 04 — Matrix Multiplication Kernels

Covers Phase 4 of the kernel roadmap:
- `naive_matmul` — no tiling, baseline
- `tiled_matmul` — SRAM tiling + autotuning
- `batched_matmul` — BMM with batch dimension

**Metric**: TFLOPS — `(2 × M × N × K × 1e-12) / (ms × 1e-3)`

In [ ]:
# ── Setup: mount Drive and clone / pull the repo ─────────────────────────────
import os
from google.colab import drive

drive.mount("/content/drive")

REPO_URL    = "https://github.com/Bhavikupadhyay/triton-kernels.git"
REPO_BRANCH = "feature/naive-matmul"
REPO_DIR    = "/content/drive/MyDrive/triton-kernels"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all
    !git -C {REPO_DIR} checkout -f {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!bash scripts/setup_colab.sh

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import triton

from kernels.matmul.naive_matmul import naive_matmul, test_naive_matmul, benchmark_naive_matmul

# Uncomment as kernels are implemented:
# from kernels.matmul.tiled_matmul import tiled_matmul, test_tiled_matmul, benchmark_tiled_matmul
# from kernels.matmul.batched_matmul import batched_matmul, test_batched_matmul, benchmark_batched_matmul

print("Imports ready")

## 1. naive_matmul

**File**: `kernels/matmul/naive_matmul.py`  
**PyTorch equivalent**: `torch.matmul(A, B)`  
**Key**: No SRAM tiling — serves as a baseline to show why tiling matters.

In [ ]:
# ── naive_matmul: Correctness ─────────────────────────────────────────────────
test_naive_matmul()

In [ ]:
# ── naive_matmul: Benchmark ───────────────────────────────────────────────────
import os
os.makedirs("benchmarks/results/matmul", exist_ok=True)

benchmark_naive_matmul.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/matmul",
)

**Interpretation — naive_matmul (T4, fp32, square M=N=K)**

**cuBLAS wins at every size except N=256, and the gap is 2–4×.** This is the expected result — naive exists to show the baseline before tiling is added.

**N=256 — Triton ahead (0.75 vs 0.62 TFLOPS).** Both are latency-dominated; the matrix is too small to saturate the GPU. Triton's launch overhead is lower here, giving it a slight edge.

**N=512 — cuBLAS jumps to 4.37 TFLOPS; Triton reaches 1.06.** This is where cuBLAS's SRAM tiling begins to pay off. It loads tiles of A and B into shared memory and reuses them across the dot products for that tile, keeping HBM traffic proportional to the tile size rather than the full row/column. Our kernel loads fresh data from HBM on every K-step with no reuse.

**N=1024 — Triton peaks at 2.48 TFLOPS.** The K-loop is long enough that each program does meaningful compute per HBM load, pushing utilization up. cuBLAS peaks here too at 5.60 TFLOPS (~69% of T4's 8.1 TFLOPS fp32 theoretical).

**N≥2048 — both drop; Triton more sharply.** Once the full A and B matrices exceed L2 (4 MB on T4 — a 1024×1024 fp32 matrix is exactly 4 MB), every load in the K-loop is a cold HBM read. Without group ordering, adjacent programs in the launch grid share no A or B tiles in L2, so each program independently re-reads its entire row and column stripe from HBM. Triton drops to 1.93 and 1.77 TFLOPS. cuBLAS drops too but less severely because its SRAM tiling limits how much data each block needs to pull from HBM per output tile.

**Absolute numbers are low for both.** T4 fp32 theoretical is 8.1 TFLOPS (Turing tensor cores only support FP16/INT8 at high throughput, not FP32). Triton peaks at 31% of theoretical; cuBLAS at 69%. Neither is close to the ceiling — matmul efficiency is dominated by how well you tile for SRAM reuse, which is exactly what tiled_matmul addresses.

## 2. tiled_matmul

**File**: `kernels/matmul/tiled_matmul.py`  
**Key**: SRAM tiling reduces HBM bandwidth. Uses `@triton.autotune` to find optimal `BLOCK_M`, `BLOCK_N`, `BLOCK_K`.

In [ ]:
# ── tiled_matmul: Correctness ────────────────────────────────────────────────
# test_tiled_matmul()

In [ ]:
# ── tiled_matmul: Benchmark ──────────────────────────────────────────────────
# benchmark_tiled_matmul.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/matmul/tiled_matmul_benchmark.png")

**Interpretation**: Add notes here.

## 3. batched_matmul

**File**: `kernels/matmul/batched_matmul.py`  
**PyTorch equivalent**: `torch.bmm(A, B)`  
**Key**: Adds a batch dimension using `tl.program_id(2)`.

In [ ]:
# ── batched_matmul: Correctness ──────────────────────────────────────────────
# test_batched_matmul()

In [ ]:
# ── batched_matmul: Benchmark ────────────────────────────────────────────────
# benchmark_batched_matmul.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/matmul/batched_matmul_benchmark.png")

**Interpretation**: Add notes here.

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
# import pandas as pd, glob
# csvs = glob.glob("benchmarks/results/matmul/*.csv")
# if csvs:
#     print(pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True).to_string(index=False))
# else:
#     print("No CSVs yet.")